# CRF Data Preparation Pipeline for Medical Dataset Name Extraction

This notebook prepares training data for a **CRF (Conditional Random Field)** baseline model that extracts dataset names from medical research articles.

It follows the same data processing pipeline as `data_preparation.ipynb` (used for GLiNER2), but outputs **BIO-tagged token sequences** instead of GLiNER2 JSONL.

| Step | Description |
|------|-------------|
| 1 | Setup and configuration |
| 2 | Parse Label Studio JSON annotations |
| 3 | Filter unlabeled documents and analyze dataset |
| 4 | Entity-aware text chunking |
| 5 | Controlled negative sampling |
| 6 | Entity-centered augmentation |
| 7 | Convert chunks to BIO-tagged token sequences |
| 8 | Train/validation/test split and save |

## 1. Setup and Configuration

In [4]:
# Install dependencies
# Uncomment the line below when running on Google Colab
%pip install -q sklearn-crfsuite spacy seqeval joblib huggingface_hub pandas matplotlib optuna
!python3 -m spacy download en_core_web_sm

Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 13.3 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [5]:
import json
import os
import pickle
import random
import re
from collections import Counter
from typing import Dict, List, Optional, Tuple

import spacy

# ============================================================
# CONFIGURATION - Modify these parameters for your experiment
# ============================================================

# --- Paths ---
DATASET_PATH = "data-annotations.json"           # Main Label Studio JSON export
OOD_DATASET_PATH = "151-eval.json"               # Out-of-distribution eval set
OUTPUT_DIR = "./crf_data"                         # Output directory for prepared data

# --- Chunking (same as GLiNER2 pipeline) ---
CHUNK_SIZE = 1500                                 # Max characters per chunk
CHUNK_OVERLAP = 300                               # Overlap between consecutive chunks

# --- Negative sampling (same as GLiNER2 pipeline) ---
NEGATIVE_SAMPLE_RATIO = 0.15                      # Keep 15% of entity-free chunks

# --- Augmentation (same as GLiNER2 pipeline) ---
AUGMENTATION_ENABLED = True
AUGMENTATION_WINDOW_SIZE = 800                    # Smaller focused windows around entities
AUGMENTATION_MAX_PER_DOC = 5                      # Max augmented chunks per document

# --- Split ratios (same as GLiNER2 pipeline) ---
TRAIN_RATIO = 0.75
VAL_RATIO = 0.10
TEST_RATIO = round(1.0 - TRAIN_RATIO - VAL_RATIO, 2)  # 0.15

# --- Reproducibility ---
SEED = 42

rng = random.Random(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

print(f"Output directory: {OUTPUT_DIR}")
print(f"Chunk size: {CHUNK_SIZE} chars, overlap: {CHUNK_OVERLAP} chars")
print(f"Negative sample ratio: {NEGATIVE_SAMPLE_RATIO}")
print(f"Augmentation: {'ON' if AUGMENTATION_ENABLED else 'OFF'}")
print(f"Split: {TRAIN_RATIO:.0%} train / {VAL_RATIO:.0%} val / {TEST_RATIO:.0%} test")
print(f"Seed: {SEED}")
print(f"OOD eval set: {OOD_DATASET_PATH}")

Output directory: ./crf_data
Chunk size: 1500 chars, overlap: 300 chars
Negative sample ratio: 0.15
Augmentation: ON
Split: 75% train / 10% val / 15% test
Seed: 42


## 2. Parse Label Studio Data

The dataset is in **Label Studio JSON** format. Each item contains:
- `data.text` - the full document text
- `annotations[].result[]` - list of annotation results, each with:
  - `value.start` / `value.end` - character offsets
  - `value.text` - the annotated span text
  - `value.labels` - list of entity type labels (e.g., `["Dataset"]`)

In [6]:
def parse_label_studio_json(filepath: str) -> List[Dict]:
    """
    Parse a Label Studio JSON export into a list of documents.
    Normalizes document text (replaces newlines/tabs with spaces).
    
    Returns list of dicts:
        {'text': str, 'entities': [{'start', 'end', 'text', 'label'}], 'is_labeled': bool}
    """
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    parsed = []
    for item in data:
        raw_text = (item.get("data") or {}).get("text", "")
        if not raw_text:
            continue

        # Normalize: replace newlines/tabs with spaces (preserves char offsets)
        text = raw_text.replace('\r\n', ' ').replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')

        entities = []
        for annot in item.get("annotations") or []:
            for r in annot.get("result") or []:
                value = r.get("value") or {}
                start = value.get("start")
                end = value.get("end")
                labels = value.get("labels") or []
                if start is not None and end is not None:
                    entity_text = text[start:end].strip()
                    if not entity_text:
                        continue
                    # Adjust offsets to match stripped text
                    offset = text[start:end].index(entity_text) if entity_text in text[start:end] else 0
                    adj_start = start + offset
                    adj_end = adj_start + len(entity_text)
                    for label in labels:
                        entities.append(
                            {"start": adj_start, "end": adj_end, "text": entity_text, "label": label}
                        )

        parsed.append({"text": text, "entities": entities, "is_labeled": len(entities) > 0})

    return parsed

In [7]:
documents = parse_label_studio_json(DATASET_PATH)

n_labeled = sum(1 for d in documents if d["is_labeled"])
n_unlabeled = sum(1 for d in documents if not d["is_labeled"])

print(f"Total documents: {len(documents)}")
print(f"  Labeled:   {n_labeled}")
print(f"  Unlabeled: {n_unlabeled}")

Total documents: 954
  Labeled:   542
  Unlabeled: 412


In [8]:
# Display 5 sample documents
print("=" * 60)
print("SAMPLE DOCUMENTS")
print("=" * 60)

for i, doc in enumerate(documents[:5]):
    print(f"\n--- Document {i+1} ---")
    print(f"Text length: {len(doc['text'])} chars")
    print(f"Text preview: {doc['text'][:200]}...")
    print(f"Entities ({len(doc['entities'])}):")
    for ent in doc["entities"][:10]:
        print(f"  [{ent['start']}:{ent['end']}] {ent['label']}: \"{ent['text']}\"")
    if len(doc["entities"]) > 10:
        print(f"  ... and {len(doc['entities']) - 10} more")

SAMPLE DOCUMENTS

--- Document 1 ---
Text length: 25176 chars
Text preview: Annotating Synapses in Large EM Datasets Reconstructing neuronal circuits at the level of synapses is a central problem in neuroscience and becoming a focus of the emerging field of connectomics. To d...
Entities (4):
  [977:993] Dataset: "EM image dataset"
  [2082:2096] Dataset: "FIB-SEM images"
  [2082:2096] Dataset: "FIB-SEM images"
  [980:994] Dataset: "image dataset."

--- Document 2 ---
Text length: 10302 chars
Text preview: Automatic Brain Tumor Segmentation with Scale Attention Network Automatic segmentation of brain tumors is an essential but challenging step for extracting quantitative imaging biomarkers for accurate ...
Entities (6):
  [8118:8128] Dataset: "BraTS 2020"
  [9214:9224] Dataset: "BraTS 2020"
  [9388:9398] Dataset: "Brats 2020"
  [274:337] Dataset: "Multimodal Brain Tumor Segmentation Challenge 2020 (BraTS 2020)"
  [1154:1164] Dataset: "BraTS 2020"
  [1486:1496] Dataset: "BraTS 2020"

---

## 3. Filter & Analyze

In [9]:
# Filter unlabeled documents
labeled_docs = [doc for doc in documents if doc["is_labeled"]]
n_removed = len(documents) - len(labeled_docs)

print(f"Filtered out {n_removed} unlabeled documents.")
print(f"Remaining labeled documents: {len(labeled_docs)}")

Filtered out 412 unlabeled documents.
Remaining labeled documents: 542


In [10]:
# Dataset statistics
label_counter = Counter()
entities_per_doc = []
text_lengths = []
entity_text_lengths = []

for doc in labeled_docs:
    text_lengths.append(len(doc["text"]))
    entities_per_doc.append(len(doc["entities"]))
    for ent in doc["entities"]:
        label_counter[ent["label"]] += 1
        entity_text_lengths.append(len(ent["text"]))

print("=" * 60)
print("DATASET STATISTICS")
print("=" * 60)
print(f"Documents:           {len(labeled_docs)}")
print(f"Total entities:      {sum(label_counter.values())}")
print(f"Label types:         {dict(label_counter)}")
print(f"Entities per doc:    mean={sum(entities_per_doc)/len(entities_per_doc):.1f}, "
      f"min={min(entities_per_doc)}, max={max(entities_per_doc)}")
print(f"Text length (chars): mean={sum(text_lengths)/len(text_lengths):.0f}, "
      f"min={min(text_lengths)}, max={max(text_lengths)}")
print(f"Entity text length:  mean={sum(entity_text_lengths)/len(entity_text_lengths):.1f}, "
      f"min={min(entity_text_lengths)}, max={max(entity_text_lengths)}")

DATASET STATISTICS
Documents:           542
Total entities:      6353
Label types:         {'Dataset': 6353}
Entities per doc:    mean=11.7, min=1, max=109
Text length (chars): mean=26383, min=4065, max=299869
Entity text length:  mean=13.9, min=2, max=194


## 4. Entity-Aware Text Chunking

Documents average ~26K characters, but spaCy and CRF models work best on shorter text. We use a sliding window approach:
- **Window size**: 1500 characters (same as GLiNER2 pipeline)
- **Overlap**: 300 characters to avoid splitting entities at boundaries
- **Sentence-boundary alignment**: tries to break at sentence ends
- **Entity mapping**: only includes entities fully contained within a chunk, with offsets adjusted to be relative to the chunk start

In [11]:
def chunk_document(doc: Dict, chunk_size: int, chunk_overlap: int) -> List[Dict]:
    """
    Split a long document into overlapping chunks.
    Maps entity spans to their new positions within each chunk.
    Entities spanning chunk boundaries are discarded.

    Returns list of dicts:
        {'text': str, 'entities': [...], 'has_entities': bool, 'source': 'chunk'}
    """
    text = doc["text"]
    entities = doc["entities"]
    text_len = len(text)
    chunks = []
    start = 0

    while start < text_len:
        end = min(start + chunk_size, text_len)

        # Try to break at a sentence boundary near the end
        if end < text_len:
            search_start = max(start, end - 200)
            for sep in [". ", ".\n", "\n\n", "\n", " "]:
                last_sep = text[search_start:end].rfind(sep)
                if last_sep != -1:
                    end = search_start + last_sep + len(sep)
                    break

        chunk_text = text[start:end]

        # Map entities to this chunk (only fully contained entities)
        chunk_entities = []
        for ent in entities:
            if ent["start"] >= start and ent["end"] <= end:
                chunk_entities.append({
                    "start": ent["start"] - start,
                    "end": ent["end"] - start,
                    "text": ent["text"],
                    "label": ent["label"],
                })

        chunks.append({
            "text": chunk_text,
            "entities": chunk_entities,
            "has_entities": len(chunk_entities) > 0,
            "source": "chunk",
        })

        if end >= text_len:
            break
        start = end - chunk_overlap

    return chunks


def chunk_all_documents(documents: List[Dict]) -> List[Dict]:
    """Chunk all documents and return flat list of chunks."""
    all_chunks = []
    for doc in documents:
        all_chunks.extend(chunk_document(doc, CHUNK_SIZE, CHUNK_OVERLAP))
    return all_chunks

In [12]:
all_chunks = chunk_all_documents(labeled_docs)

n_pos = sum(1 for c in all_chunks if c["has_entities"])
n_neg = sum(1 for c in all_chunks if not c["has_entities"])
total_ents = sum(len(c["entities"]) for c in all_chunks)

print(f"Total chunks: {len(all_chunks)}")
print(f"  With entities:    {n_pos} ({n_pos/len(all_chunks):.1%})")
print(f"  Without entities: {n_neg} ({n_neg/len(all_chunks):.1%})")
print(f"  Total entity mentions: {total_ents}")

Total chunks: 12730
  With entities:    3391 (26.6%)
  Without entities: 9339 (73.4%)
  Total entity mentions: 7948


## 5. Controlled Negative Sampling

After chunking, most chunks have no entities. We keep ALL entity-containing chunks but subsample entity-free chunks at the configured ratio to balance the training data.

In [13]:
def apply_negative_sampling(chunks: List[Dict], ratio: float) -> Tuple[List[Dict], Dict]:
    """
    Keep all entity-containing chunks.
    Subsample entity-free chunks at the given ratio.
    """
    positive = [c for c in chunks if c["has_entities"]]
    negative = [c for c in chunks if not c["has_entities"]]

    n_neg_keep = max(1, int(len(negative) * ratio))
    sampled_neg = rng.sample(negative, min(n_neg_keep, len(negative)))

    balanced = positive + sampled_neg
    rng.shuffle(balanced)

    stats = {
        "positive_chunks": len(positive),
        "total_negative_chunks": len(negative),
        "sampled_negative_chunks": len(sampled_neg),
        "total_after_sampling": len(balanced),
    }
    return balanced, stats


balanced_chunks, neg_stats = apply_negative_sampling(all_chunks, NEGATIVE_SAMPLE_RATIO)

print(f"Negative Sampling Results:")
print(f"  Positive chunks (kept all):     {neg_stats['positive_chunks']}")
print(f"  Negative chunks (before):       {neg_stats['total_negative_chunks']}")
print(f"  Negative chunks (after):        {neg_stats['sampled_negative_chunks']}")
print(f"  Total after sampling:           {neg_stats['total_after_sampling']}")

Negative Sampling Results:
  Positive chunks (kept all):     3391
  Negative chunks (before):       9339
  Negative chunks (after):        1400
  Total after sampling:           4791


## 6. Entity-Centered Augmentation

Create additional smaller windows (800 chars) centered on entity mentions. This oversamples entity-rich regions, giving the CRF more positive training examples.

In [14]:
def create_entity_centered_chunks(doc: Dict) -> List[Dict]:
    """
    Create smaller, focused chunks centered on entity mentions.
    This oversamples entity-rich regions as a form of augmentation.
    """
    if not doc["entities"]:
        return []

    text = doc["text"]
    text_len = len(text)
    half_window = AUGMENTATION_WINDOW_SIZE // 2
    aug_chunks = []
    used_centers = set()

    shuffled_entities = list(doc["entities"])
    rng.shuffle(shuffled_entities)

    for ent in shuffled_entities:
        if len(aug_chunks) >= AUGMENTATION_MAX_PER_DOC:
            break

        center = (ent["start"] + ent["end"]) // 2

        # Skip if too close to an already-used center
        if any(abs(center - uc) < half_window // 2 for uc in used_centers):
            continue
        used_centers.add(center)

        win_start = max(0, center - half_window)
        win_end = min(text_len, center + half_window)

        # Try to align to sentence boundaries
        for sep in [". ", "\n"]:
            idx = text[win_start:min(win_start + 100, center)].find(sep)
            if idx != -1:
                win_start = win_start + idx + len(sep)
                break

        chunk_text = text[win_start:win_end]

        chunk_entities = []
        for e in doc["entities"]:
            if e["start"] >= win_start and e["end"] <= win_end:
                chunk_entities.append({
                    "start": e["start"] - win_start,
                    "end": e["end"] - win_start,
                    "text": e["text"],
                    "label": e["label"],
                })

        if chunk_entities:
            aug_chunks.append({
                "text": chunk_text,
                "entities": chunk_entities,
                "has_entities": True,
                "source": "augmentation",
            })

    return aug_chunks


if AUGMENTATION_ENABLED:
    aug_chunks = []
    for doc in labeled_docs:
        aug_chunks.extend(create_entity_centered_chunks(doc))

    print(f"Entity-centered augmentation chunks created: {len(aug_chunks)}")
    print(f"  Avg entities per aug chunk: {sum(len(c['entities']) for c in aug_chunks)/max(len(aug_chunks),1):.2f}")

    combined_chunks = balanced_chunks + aug_chunks
    rng.shuffle(combined_chunks)

    print(f"\nTotal chunks after augmentation: {len(combined_chunks)}")
    print(f"  From regular chunking: {len(balanced_chunks)}")
    print(f"  From augmentation:     {len(aug_chunks)}")
else:
    combined_chunks = balanced_chunks
    print("Augmentation disabled. Using balanced chunks only.")
    print(f"Total chunks: {len(combined_chunks)}")

Entity-centered augmentation chunks created: 2054
  Avg entities per aug chunk: 2.18

Total chunks after augmentation: 6845
  From regular chunking: 4791
  From augmentation:     2054


## 7. Convert Chunks to BIO-Tagged Sequences

This is where the CRF pipeline diverges from GLiNER2. Instead of converting to GLiNER2 JSONL format, we:

1. **Tokenize** each chunk with spaCy to get tokens with character offsets and POS tags
2. **Align** entity character spans (already in the chunk data) to token boundaries
3. **Tag** tokens using the BIO scheme: `B-Dataset` (beginning), `I-Dataset` (inside), `O` (outside)
4. **Split** each chunk into sentences for CRF training (CRFs work best on sentence-length sequences)

### BIO Tagging Scheme
```
Token:  We    used    the    MIMIC   -    III    dataset    for    training
Tag:    O     O       O      B-Dataset I-Dataset I-Dataset  O       O      O
```

In [15]:
def chunk_to_bio(chunk: Dict, nlp) -> Tuple[List[Dict], Dict]:
    """
    Convert a single chunk to BIO-tagged token sequences.
    
    Returns:
        sentences: list of {'tokens': [...], 'pos_tags': [...], 'bio_tags': [...]} per sentence
        chunk_data: {'tokens': [...], 'pos_tags': [...], 'bio_tags': [...], 'gold_entities': [...]}
                    for chunk-level evaluation later
    """
    text = chunk["text"]
    entities = chunk["entities"]
    
    doc = nlp(text)
    
    # Initialize all tags as 'O'
    chunk_tags = ['O'] * len(doc)
    
    # For each entity, find overlapping tokens and assign BIO tags
    for ent in entities:
        ent_start = ent["start"]
        ent_end = ent["end"]
        first_token = True
        
        for i, tok in enumerate(doc):
            tok_start = tok.idx
            tok_end = tok.idx + len(tok.text)
            
            # Token overlaps with entity span
            if tok_start < ent_end and tok_end > ent_start:
                if first_token:
                    chunk_tags[i] = f'B-{ent["label"]}'
                    first_token = False
                else:
                    # Only set I- tag if not already tagged as B- by another entity
                    if chunk_tags[i] == 'O':
                        chunk_tags[i] = f'I-{ent["label"]}'
    
    # Collect chunk-level data
    chunk_tokens = [tok.text for tok in doc]
    chunk_pos = [tok.pos_ for tok in doc]
    gold_entities = [ent["text"] for ent in entities]
    
    chunk_data = {
        'tokens': chunk_tokens,
        'pos_tags': chunk_pos,
        'bio_tags': list(chunk_tags),
        'gold_entities': gold_entities,
    }
    
    # Split into sentences
    sentences = []
    for sent in doc.sents:
        sent_tokens = []
        sent_pos = []
        sent_tags = []
        for tok in sent:
            sent_tokens.append(tok.text)
            sent_pos.append(tok.pos_)
            sent_tags.append(chunk_tags[tok.i])
        
        if sent_tokens:  # skip empty sentences
            sentences.append({
                'tokens': sent_tokens,
                'pos_tags': sent_pos,
                'bio_tags': sent_tags,
            })
    
    return sentences, chunk_data


def convert_all_chunks(chunks: List[Dict], nlp) -> Tuple[List[List[Dict]], List[Dict]]:
    """
    Convert all chunks to BIO-tagged sequences.
    
    Returns:
        chunk_sentence_groups: list of (list of sentence dicts) - one group per chunk
        chunk_data_list: list of chunk-level dicts for evaluation
    """
    chunk_sentence_groups = []
    chunk_data_list = []
    
    for i, chunk in enumerate(chunks):
        if (i + 1) % 500 == 0:
            print(f"  Processing chunk {i+1}/{len(chunks)}...")
        
        sentences, chunk_data = chunk_to_bio(chunk, nlp)
        chunk_sentence_groups.append(sentences)
        chunk_data_list.append(chunk_data)
    
    return chunk_sentence_groups, chunk_data_list

In [16]:
print(f"Converting {len(combined_chunks)} chunks to BIO-tagged sequences...")
chunk_sentence_groups, chunk_data_list = convert_all_chunks(combined_chunks, nlp)

# Flatten for statistics
all_sentences = [s for group in chunk_sentence_groups for s in group]

# Tag distribution
tag_counts = Counter()
for sent in all_sentences:
    tag_counts.update(sent['bio_tags'])

total_tokens = sum(tag_counts.values())
print(f"\nConversion complete!")
print(f"Total sentences: {len(all_sentences)}")
print(f"Total tokens: {total_tokens}")
print(f"\nBIO tag distribution:")
for tag, count in sorted(tag_counts.items()):
    print(f"  {tag:<15} {count:>8} ({count/total_tokens:.2%})")

# Sentence length stats
sent_lens = [len(s['tokens']) for s in all_sentences]
print(f"\nSentence length: mean={sum(sent_lens)/len(sent_lens):.1f}, "
      f"min={min(sent_lens)}, max={max(sent_lens)}, "
      f"median={sorted(sent_lens)[len(sent_lens)//2]}")

Converting 6845 chunks to BIO-tagged sequences...
  Processing chunk 500/6845...
  Processing chunk 1000/6845...
  Processing chunk 1500/6845...
  Processing chunk 2000/6845...
  Processing chunk 2500/6845...
  Processing chunk 3000/6845...
  Processing chunk 3500/6845...
  Processing chunk 4000/6845...
  Processing chunk 4500/6845...
  Processing chunk 5000/6845...
  Processing chunk 5500/6845...
  Processing chunk 6000/6845...
  Processing chunk 6500/6845...

Conversion complete!
Total sentences: 65249
Total tokens: 1559560

BIO tag distribution:
  B-Dataset          12335 (0.79%)
  I-Dataset          18785 (1.20%)
  O                1528440 (98.00%)

Sentence length: mean=23.9, min=1, max=349, median=22


In [17]:
# Display 5 sample converted sentences with BIO tags
print("=" * 60)
print("SAMPLE BIO-TAGGED SENTENCES")
print("=" * 60)

sample_count = 0
for sent in all_sentences:
    # Show sentences that contain entities
    if any(t != 'O' for t in sent['bio_tags']):
        print(f"\n--- Sample {sample_count + 1} ---")
        print(f"{'Token':<25} {'POS':<8} {'BIO Tag'}")
        print("-" * 50)
        for tok, pos, tag in zip(sent['tokens'], sent['pos_tags'], sent['bio_tags']):
            marker = " <<" if tag != 'O' else ""
            print(f"{tok:<25} {pos:<8} {tag}{marker}")
        sample_count += 1
        if sample_count >= 5:
            break

SAMPLE BIO-TAGGED SENTENCES

--- Sample 1 ---
Token                     POS      BIO Tag
--------------------------------------------------
Memory                    NOUN     O
Efficient                 PROPN    O
3D                        NOUN     O
U                         PROPN    O
-                         PROPN    O
Net                       PROPN    O
with                      ADP      O
Reversible                PROPN    O
Mobile                    PROPN    O
Inverted                  PROPN    O
Bottlenecks               PROPN    O
for                       ADP      O
Brain                     PROPN    O
Tumor                     PROPN    O
Segmentation              PROPN    O
We                        PRON     O
propose                   VERB     O
combining                 VERB     O
memory                    NOUN     O
saving                    NOUN     O
techniques                NOUN     O
with                      ADP      O
traditional               ADJ      O
U        

In [18]:
# Validation: decode BIO tags back to entity strings and compare with originals
def bio_to_entity_strings(tokens: List[str], tags: List[str]) -> List[str]:
    """Decode BIO tags back to entity strings."""
    entities = []
    current_tokens = []
    
    for tok, tag in zip(tokens, tags):
        if tag.startswith('B-'):
            if current_tokens:
                entities.append(' '.join(current_tokens))
            current_tokens = [tok]
        elif tag.startswith('I-') and current_tokens:
            current_tokens.append(tok)
        else:
            if current_tokens:
                entities.append(' '.join(current_tokens))
                current_tokens = []
    
    if current_tokens:
        entities.append(' '.join(current_tokens))
    
    return entities


# Validate on 100 random chunks
n_validate = min(100, len(chunk_data_list))
indices = rng.sample(range(len(chunk_data_list)), n_validate)

match_count = 0
total_checked = 0
mismatch_examples = []

for idx in indices:
    cd = chunk_data_list[idx]
    decoded = bio_to_entity_strings(cd['tokens'], cd['bio_tags'])
    gold = cd['gold_entities']
    
    # Normalize for comparison
    decoded_norm = set(e.strip().lower() for e in decoded)
    gold_norm = set(e.strip().lower() for e in gold)
    
    if decoded_norm == gold_norm:
        match_count += 1
    else:
        if len(mismatch_examples) < 5:
            mismatch_examples.append({
                'gold': gold_norm,
                'decoded': decoded_norm,
                'missing': gold_norm - decoded_norm,
                'extra': decoded_norm - gold_norm,
            })
    total_checked += 1

print(f"BIO Conversion Validation ({n_validate} random chunks):")
print(f"  Exact match: {match_count}/{total_checked} ({match_count/total_checked:.1%})")

if mismatch_examples:
    print(f"\nMismatch examples (first {len(mismatch_examples)}):")
    for i, ex in enumerate(mismatch_examples):
        print(f"  Example {i+1}:")
        print(f"    Gold:    {ex['gold']}")
        print(f"    Decoded: {ex['decoded']}")
        print(f"    Missing: {ex['missing']}")
        print(f"    Extra:   {ex['extra']}")

BIO Conversion Validation (100 random chunks):
  Exact match: 77/100 (77.0%)

Mismatch examples (first 5):
  Example 1:
    Gold:    {'vascusynth'}
    Decoded: {'vascusynth-3', 'vascusynth-2', 'vascusynth-1'}
    Missing: {'vascusynth'}
    Extra:   {'vascusynth-3', 'vascusynth-2', 'vascusynth-1'}
  Example 2:
    Gold:    {'mm-whs dataset'}
    Decoded: {'mm - whs dataset'}
    Missing: {'mm-whs dataset'}
    Extra:   {'mm - whs dataset'}
  Example 3:
    Gold:    {'us-case', 'cca-us'}
    Decoded: {'cca - us', 'us - case'}
    Missing: {'us-case', 'cca-us'}
    Extra:   {'cca - us', 'us - case'}
  Example 4:
    Gold:    {'mimic-cxr training set'}
    Decoded: {'mimic - cxr training set'}
    Missing: {'mimic-cxr training set'}
    Extra:   {'mimic - cxr training set'}
  Example 5:
    Gold:    {'mendeley-v2', 'mimic-cxr', 'mimic-cxr dataset'}
    Decoded: {'mimic - cxr', 'mimic - cxr dataset', 'mendeley - v2'}
    Missing: {'mimic-cxr', 'mendeley-v2', 'mimic-cxr dataset'}
    Extra

## 8. Train/Validation/Test Split & Save

Data is shuffled and split at the **chunk level** (not sentence level) to ensure no sentences from the same chunk appear in different splits:
- **Training set** (75%) - used for CRF model training
- **Validation set** (10%) - used for hyperparameter tuning
- **Test set** (15%) - held-out for final evaluation

In [19]:
# Pair sentence groups with chunk data for splitting
paired = list(zip(chunk_sentence_groups, chunk_data_list))
rng.shuffle(paired)

n_total = len(paired)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VAL_RATIO)

train_paired = paired[:n_train]
val_paired = paired[n_train:n_train + n_val]
test_paired = paired[n_train + n_val:]

# Separate sentence groups and chunk data
train_sentence_groups, train_chunks = zip(*train_paired) if train_paired else ([], [])
val_sentence_groups, val_chunks = zip(*val_paired) if val_paired else ([], [])
test_sentence_groups, test_chunks = zip(*test_paired) if test_paired else ([], [])

# Flatten sentence groups into flat lists
train_sentences = [s for group in train_sentence_groups for s in group]
val_sentences = [s for group in val_sentence_groups for s in group]
test_sentences = [s for group in test_sentence_groups for s in group]

# Convert chunk data tuples back to lists
train_chunks = list(train_chunks)
val_chunks = list(val_chunks)
test_chunks = list(test_chunks)

print(f"Split results:")
print(f"{'Split':<12} {'Chunks':>8} {'Sentences':>10} {'Tokens':>10}")
print("-" * 45)
for name, sents, chunks in [
    ("Train", train_sentences, train_chunks),
    ("Val", val_sentences, val_chunks),
    ("Test", test_sentences, test_chunks),
]:
    n_tokens = sum(len(s['tokens']) for s in sents)
    print(f"{name:<12} {len(chunks):>8} {len(sents):>10} {n_tokens:>10}")

total_chunks = len(train_chunks) + len(val_chunks) + len(test_chunks)
total_sents = len(train_sentences) + len(val_sentences) + len(test_sentences)
print("-" * 45)
print(f"{'Total':<12} {total_chunks:>8} {total_sents:>10}")

Split results:
Split          Chunks  Sentences     Tokens
---------------------------------------------
Train            5133      49027    1173686
Val               684       6474     153384
Test             1028       9748     232490
---------------------------------------------
Total            6845      65249


In [20]:
# BIO tag distribution per split
print("\nBIO tag distribution per split:")
print(f"{'Split':<12} {'B-Dataset':>10} {'I-Dataset':>10} {'O':>10} {'Entity mentions':>16}")
print("-" * 65)
for name, sents, chunks in [
    ("Train", train_sentences, train_chunks),
    ("Val", val_sentences, val_chunks),
    ("Test", test_sentences, test_chunks),
]:
    tc = Counter()
    for s in sents:
        tc.update(s['bio_tags'])
    n_ents = sum(len(c['gold_entities']) for c in chunks)
    print(f"{name:<12} {tc.get('B-Dataset', 0):>10} {tc.get('I-Dataset', 0):>10} {tc.get('O', 0):>10} {n_ents:>16}")


BIO tag distribution per split:
Split         B-Dataset  I-Dataset          O  Entity mentions
-----------------------------------------------------------------
Train              9139      14058    1150489             9216
Val                1301       1891     150192             1314
Test               1895       2836     227759             1900


In [21]:
# Save to crf_data/ directory
for name, sents, chunks in [
    ("train", train_sentences, train_chunks),
    ("val", val_sentences, val_chunks),
    ("test", test_sentences, test_chunks),
]:
    sent_path = os.path.join(OUTPUT_DIR, f"{name}_sentences.pkl")
    chunk_path = os.path.join(OUTPUT_DIR, f"{name}_chunks.pkl")
    
    with open(sent_path, 'wb') as f:
        pickle.dump(sents, f)
    with open(chunk_path, 'wb') as f:
        pickle.dump(chunks, f)
    
    print(f"Saved {name}_sentences.pkl ({len(sents)} sentences, {os.path.getsize(sent_path)/1024:.1f} KB)")
    print(f"Saved {name}_chunks.pkl ({len(chunks)} chunks, {os.path.getsize(chunk_path)/1024:.1f} KB)")

print("\nMetadata will be saved after OOD processing (Section 9).")


Saved train_sentences.pkl (49027 sentences, 14316.4 KB)
Saved train_chunks.pkl (5133 chunks, 17170.6 KB)
Saved val_sentences.pkl (6474 sentences, 1856.4 KB)
Saved val_chunks.pkl (684 chunks, 2202.3 KB)
Saved test_sentences.pkl (9748 sentences, 2844.4 KB)
Saved test_chunks.pkl (1028 chunks, 2692.2 KB)

Saved metadata.json (0.6 KB)


## 9. Process Out-of-Distribution (OOD) Evaluation Set

The OOD set (`151-eval.json`) is processed with the **same full pipeline** as the main dataset:
- **Same chunking** -- entity-aware sliding window (1500 chars, 300 overlap)
- **Same negative sampling** -- 15% ratio for consistency with val/test splits
- **Same augmentation** -- entity-centered chunks for balanced entity representation
- **No split** -- entire set used for evaluation

This ensures fair within-CRF comparison across val, test, and OOD splits.

In [ ]:
# Parse OOD evaluation set
ood_docs = parse_label_studio_json(OOD_DATASET_PATH)

n_ood_labeled = sum(1 for d in ood_docs if d['is_labeled'])
n_ood_unlabeled = sum(1 for d in ood_docs if not d['is_labeled'])
total_ood_ents = sum(len(d['entities']) for d in ood_docs)

print(f"OOD evaluation set: {OOD_DATASET_PATH}")
print(f"  Total documents: {len(ood_docs)}")
print(f"  Labeled:   {n_ood_labeled}")
print(f"  Unlabeled: {n_ood_unlabeled}")
print(f"  Total entity annotations: {total_ood_ents}")

# Text length comparison
ood_lengths = [len(d['text']) for d in ood_docs]
print(f"  Text length (chars): mean={sum(ood_lengths)/len(ood_lengths):.0f}, "
      f"min={min(ood_lengths)}, max={max(ood_lengths)}")

# Display 5 sample OOD documents
print("\n" + "=" * 60)
print("SAMPLE OOD DOCUMENTS")
print("=" * 60)
for i, doc in enumerate(ood_docs[:5]):
    print(f"\n--- OOD Document {i+1} ---")
    print(f"Text length: {len(doc['text'])} chars")
    print(f"Text preview: {doc['text'][:150]}...")
    print(f"Entities ({len(doc['entities'])}):") 
    for ent in doc['entities'][:5]:
        print(f"  [{ent['start']}:{ent['end']}] {ent['label']}: \"{ent['text']}\"")
    if len(doc['entities']) > 5:
        print(f"  ... and {len(doc['entities']) - 5} more")

In [ ]:
# Chunk OOD documents
ood_docs_labeled = [doc for doc in ood_docs if doc['is_labeled']]
ood_all_chunks = chunk_all_documents(ood_docs_labeled)

ood_n_pos = sum(1 for c in ood_all_chunks if c['has_entities'])
ood_n_neg = sum(1 for c in ood_all_chunks if not c['has_entities'])

print(f"OOD raw chunks: {len(ood_all_chunks)}")
print(f"  With entities:    {ood_n_pos} ({ood_n_pos/len(ood_all_chunks):.1%})")
print(f"  Without entities: {ood_n_neg} ({ood_n_neg/len(ood_all_chunks):.1%})")
print(f"  Total entity mentions: {sum(len(c['entities']) for c in ood_all_chunks)}")

# Apply same negative sampling as main set
ood_balanced_chunks, ood_neg_stats = apply_negative_sampling(ood_all_chunks, NEGATIVE_SAMPLE_RATIO)

print(f"\nOOD Negative Sampling:")
print(f"  Positive chunks (kept all):     {ood_neg_stats['positive_chunks']}")
print(f"  Negative chunks (before):       {ood_neg_stats['total_negative_chunks']}")
print(f"  Negative chunks (after):        {ood_neg_stats['sampled_negative_chunks']}")
print(f"  Total after sampling:           {ood_neg_stats['total_after_sampling']}")

# Apply same entity-centered augmentation as main set
if AUGMENTATION_ENABLED:
    ood_aug_chunks = []
    for doc in ood_docs_labeled:
        ood_aug_chunks.extend(create_entity_centered_chunks(doc))

    print(f"\nOOD Entity-Centered Augmentation:")
    print(f"  Augmentation chunks created: {len(ood_aug_chunks)}")
    print(f"  Avg entities per aug chunk: {sum(len(c['entities']) for c in ood_aug_chunks)/max(len(ood_aug_chunks),1):.2f}")

    ood_combined_chunks = ood_balanced_chunks + ood_aug_chunks
    rng.shuffle(ood_combined_chunks)

    print(f"\nOOD Total chunks after augmentation: {len(ood_combined_chunks)}")
    print(f"  From regular chunking: {len(ood_balanced_chunks)}")
    print(f"  From augmentation:     {len(ood_aug_chunks)}")
else:
    ood_combined_chunks = ood_balanced_chunks
    print(f"\nAugmentation disabled. OOD chunks: {len(ood_combined_chunks)}")


In [ ]:
# Convert OOD chunks to BIO-tagged sequences
print(f"Converting {len(ood_combined_chunks)} OOD chunks to BIO-tagged sequences...")
ood_sentence_groups, ood_chunks_data = convert_all_chunks(ood_combined_chunks, nlp)

ood_sentences_all = [s for group in ood_sentence_groups for s in group]

# Tag distribution
ood_tag_counts = Counter()
for sent in ood_sentences_all:
    ood_tag_counts.update(sent['bio_tags'])

ood_total_tokens = sum(ood_tag_counts.values())
print(f"\nOOD conversion complete!")
print(f"Total sentences: {len(ood_sentences_all)}")
print(f"Total tokens: {ood_total_tokens}")
print(f"\nBIO tag distribution (OOD):")
for tag, count in sorted(ood_tag_counts.items()):
    print(f"  {tag:<15} {count:>8} ({count/ood_total_tokens:.2%})")

# Display 5 sample OOD sentences with entities
print("\n" + "=" * 60)
print("SAMPLE OOD BIO-TAGGED SENTENCES")
print("=" * 60)
sample_count = 0
for sent in ood_sentences_all:
    if any(t != 'O' for t in sent['bio_tags']):
        print(f"\n--- OOD Sample {sample_count + 1} ---")
        print(f"{'Token':<25} {'POS':<8} {'BIO Tag'}")
        print("-" * 50)
        for tok, pos, tag in zip(sent['tokens'], sent['pos_tags'], sent['bio_tags']):
            marker = ' <<' if tag != 'O' else ''
            print(f"{tok:<25} {pos:<8} {tag}{marker}")
        sample_count += 1
        if sample_count >= 5:
            break


In [ ]:
# Save OOD data
ood_sent_path = os.path.join(OUTPUT_DIR, 'ood_sentences.pkl')
ood_chunk_path = os.path.join(OUTPUT_DIR, 'ood_chunks.pkl')

with open(ood_sent_path, 'wb') as f:
    pickle.dump(ood_sentences_all, f)
with open(ood_chunk_path, 'wb') as f:
    pickle.dump(list(ood_chunks_data), f)

print(f"Saved ood_sentences.pkl ({len(ood_sentences_all)} sentences, {os.path.getsize(ood_sent_path)/1024:.1f} KB)")
print(f"Saved ood_chunks.pkl ({len(ood_chunks_data)} chunks, {os.path.getsize(ood_chunk_path)/1024:.1f} KB)")

# Save metadata (after all splits including OOD are processed)
metadata = {
    "source_file": DATASET_PATH,
    "ood_source_file": OOD_DATASET_PATH,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "negative_sample_ratio": NEGATIVE_SAMPLE_RATIO,
    "augmentation_enabled": AUGMENTATION_ENABLED,
    "augmentation_window_size": AUGMENTATION_WINDOW_SIZE,
    "augmentation_max_per_doc": AUGMENTATION_MAX_PER_DOC,
    "train_ratio": TRAIN_RATIO,
    "val_ratio": VAL_RATIO,
    "test_ratio": TEST_RATIO,
    "seed": SEED,
    "spacy_model": "en_core_web_sm",
    "bio_labels": ["O", "B-Dataset", "I-Dataset"],
    "splits": {
        "train": {"chunks": len(train_chunks), "sentences": len(train_sentences)},
        "val": {"chunks": len(val_chunks), "sentences": len(val_sentences)},
        "test": {"chunks": len(test_chunks), "sentences": len(test_sentences)},
        "ood": {"chunks": len(ood_chunks_data), "sentences": len(ood_sentences_all)},
    },
    "ood_processing": {
        "negative_sampling": True,
        "augmentation": AUGMENTATION_ENABLED,
        "note": "Same pipeline as main set (neg sampling + augmentation) for consistent comparison",
    },
    "total_documents": len(labeled_docs),
    "ood_documents": len(ood_docs_labeled),
}

meta_path = os.path.join(OUTPUT_DIR, "metadata.json")
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)
print(f"\nSaved metadata.json ({os.path.getsize(meta_path)/1024:.1f} KB)")


In [22]:
# Final summary
print("=" * 60)
print("PIPELINE COMPLETE")
print("=" * 60)
print(f"\nSource: {DATASET_PATH} ({len(labeled_docs)} labeled documents)")
print(f"OOD:    {OOD_DATASET_PATH} ({len(ood_docs_labeled)} labeled documents)")
print(f"Output: {OUTPUT_DIR}/")
print(f"\nFiles generated:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    print(f"  {fname:<30} {os.path.getsize(fpath)/1024:>8.1f} KB")
print(f"\nSplit summary:")
print(f"  Train: {len(train_chunks)} chunks, {len(train_sentences)} sentences")
print(f"  Val:   {len(val_chunks)} chunks, {len(val_sentences)} sentences")
print(f"  Test:  {len(test_chunks)} chunks, {len(test_sentences)} sentences")
print(f"  OOD:   {len(ood_chunks_data)} chunks, {len(ood_sentences_all)} sentences")
print(f"\nProcessing: All splits use same pipeline (chunking + neg sampling + augmentation)")
print(f"Ready for CRF training with crf_model.ipynb")


PIPELINE COMPLETE

Source: data-annotations.json (542 labeled documents)
Output: ./crf_data/

Files generated:
  metadata.json                       0.6 KB
  test_chunks.pkl                  2692.2 KB
  test_sentences.pkl               2844.4 KB
  train_chunks.pkl                17170.6 KB
  train_sentences.pkl             14316.4 KB
  val_chunks.pkl                   2202.3 KB
  val_sentences.pkl                1856.4 KB

Split summary:
  Train: 5133 chunks, 49027 sentences
  Val:   684 chunks, 6474 sentences
  Test:  1028 chunks, 9748 sentences

Ready for CRF training with crf_model.ipynb
